In [1]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import build_preprocessor

MASTER_DF_PATH = Path("../data/processed/master_df_feature_engineered.parquet")
master_df = pd.read_parquet(MASTER_DF_PATH)

In [2]:
# look at the dataframe
master_df.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,part_of_day_purchase,product_name_length,days_since_prior_order_missing
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,soft drinks,beverages,morning,4,1
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,soy lactosefree,dairy eggs,morning,39,1
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,popcorn jerky,snacks,morning,19,1
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,popcorn jerky,snacks,morning,26,1
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,paper goods,household,morning,32,1


In [3]:
# move reordered to the end
col_to_move = master_df.pop("reordered")
master_df.insert(len(master_df.columns), "reordered", col_to_move)

In [4]:
# choose a 1mln. master df sample
master_df_sample = master_df.sample(n=1_000_000, random_state=42)

In [5]:
# get unique users
unique_users = master_df_sample["user_id"].unique()

# split users (80% of users in train, 20% in test)
# now the model never sees a user_id in the test set that is in the training set --> better for machine learning because results are more realistic
from sklearn.model_selection import train_test_split
train_users, test_users = train_test_split(unique_users, test_size=0.2, random_state=42)

# filter rows
train_df = master_df_sample[master_df_sample.user_id.isin(train_users)]
test_df = master_df_sample[master_df_sample.user_id.isin(test_users)]

# remove leakage columns
drop_cols = ["order_id", "product_name", "eval_set", "user_id", "product_id"]
train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)

# create X/y
X_train = train_df.drop("reordered", axis=1)
y_train = train_df["reordered"]

X_test = test_df.drop("reordered", axis=1)
y_test = test_df["reordered"]

In [6]:
# select numerical and categorical features
num_features = ["order_number", "days_since_prior_order", "add_to_cart_order", "product_name_length"]
cat_features = ["aisle", "department", "part_of_day_purchase"]

In [7]:
# creating pipelines
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

preprocessor = build_preprocessor(num_features, cat_features)

pipelines = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced"))
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier())
    ])
}

In [8]:
# train models and calculate metrics
import joblib
from sklearn.metrics import (accuracy_score, precision_score, recall_score, roc_auc_score)

results = {}

for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_probs = pipeline.predict_proba(X_test)[:, 1]
        
    results[model_name] = {
        "Accuracy Score": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "ROC AUC Score": roc_auc_score(y_test, y_probs)
    }
    
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [9]:
# save metrics
model_metrics_df = pd.DataFrame(results).T.round(3)
model_metrics_df.to_csv("../data/processed/model_metrics.csv")